# Efficient PageRank: Block-Stripe & MapReduce

## Learning Objectives

1. **Explain** why the naive power iteration is infeasible at web scale
2. **Describe** the block-stripe algorithm for memory-efficient PageRank
3. **Formulate** PageRank as a MapReduce job
4. **Estimate** I/O cost and memory requirements for web-scale computation

## The Scale Challenge

The web has ~$5 \times 10^{10}$ pages and ~$10^{12}$ links (hyperlinks).

**Naive power iteration** requires:
- Transition matrix $M$: $n^2$ entries = $10^{20}$ floats — impossible
- Even sparse $M$: $10^{12}$ entries × 8 bytes = **8 TB** per iteration just for $M$

**Key insight:** $M$ is very sparse (average ~20 out-links per page) and must be stored as a list of $(\text{source}, \text{dest})$ triples.

**Goal:** perform each power iteration in a **single pass** over $M$, with only $O(n)$ memory for the rank vector.

## Block-Stripe Algorithm

Partition the $n \times n$ matrix $M$ into $k$ **horizontal stripes**, each covering rows $[r_i, r_{i+1})$:

$$M = \begin{bmatrix} M_0 \\ M_1 \\ \vdots \\ M_{k-1} \end{bmatrix}$$

**Algorithm (one iteration):**
1. Load rank vector $\mathbf{r}$ into memory (fits: $n$ floats)
2. For each stripe $M_i$:
   - Read $M_i$ from disk
   - Compute $M_i \mathbf{r}$ → updates for rows in stripe $i$
   - Write result chunk to disk
3. Combine result chunks → new $\mathbf{r}'$

**Memory per step:** $n$ floats (rank) + size of one stripe ($m/k$ entries)

In [1]:
from collections import defaultdict
def pagerank_block_stripe(out_edges_list, n_nodes, beta=0.85, k=4, n_iter=30):
    """
    Simulate block-stripe PageRank.
    out_edges_list: list of (src, dst) pairs (sparse matrix)
    n_nodes: number of nodes
    k: number of stripes
    """
    # Partition edges into stripes by destination node
    stripe_size = (n_nodes + k - 1) // k
    stripes = [[] for _ in range(k)]
    out_deg = defaultdict(int)
    for src, dst in out_edges_list:
        stripe_idx = dst // stripe_size
        stripes[stripe_idx].append((src, dst))
        out_deg[src] += 1

    r = [1.0 / n_nodes] * n_nodes
    dangling_nodes = [i for i in range(n_nodes) if out_deg[i] == 0]

    for iteration in range(n_iter):
        dangling_sum = sum(r[i] for i in dangling_nodes)
        base = (beta * dangling_sum + (1 - beta)) / n_nodes
        new_r = [0.0] * n_nodes
        # Process one stripe at a time
        for stripe in stripes:
            for src, dst in stripe:
                new_r[dst] += beta * r[src] / out_deg[src]
        for i in range(n_nodes): new_r[i] += base
        diff = sum(abs(new_r[i]-r[i]) for i in range(n_nodes))
        r = new_r
        if diff < 1e-8: print(f"Converged at iteration {iteration+1}"); break
    return r

# Demo on small graph
edges = [(0,1),(0,2),(1,3),(2,1),(2,3),(3,0)]
n = 4
r = pagerank_block_stripe(edges, n, beta=0.85, k=2)
print("Block-stripe PageRank:")
for i,rank in enumerate(r):
    print(f"  node {i}: {rank:.4f}")
print(f"Total: {sum(r):.4f}")

Block-stripe PageRank:
  node 0: 0.2972
  node 1: 0.2334
  node 2: 0.1638
  node 3: 0.3056
Total: 1.0000


## MapReduce Formulation

**Map function:** for each edge $(u, v)$ with $d_u$ out-links and current rank $r_u$:
$$\text{emit}(v,\; \beta \cdot r_u / d_u)$$

Also emit the adjacency list to preserve the graph structure.

**Reduce function:** for each node $v$, sum all contributions:
$$r_v^{\text{new}} = \frac{1-\beta}{n} + \sum_{u \to v} \frac{\beta \cdot r_u}{d_u}$$

**Two-pass per iteration:**
- Pass 1: emit (node, rank contribution) pairs
- Pass 2: group by node, sum contributions

**Communication cost:** $O(m)$ per iteration (one message per edge)

In [2]:
def mapreduce_pagerank_step(edges, rank, n, beta=0.85):
    """Simulate one MapReduce PageRank iteration."""
    ""
    out_deg = defaultdict(int)
    for u,v in edges: out_deg[u] += 1
    dangling = [u for u in set(u for u,v in edges) | set(v for u,v in edges) if out_deg[u]==0]
    dangling_sum = sum(rank.get(i,0) for i in range(n) if out_deg.get(i,0)==0)

    # Map phase
    intermediate = defaultdict(float)
    for u,v in edges:
        intermediate[v] += beta * rank.get(u, 1.0/n) / out_deg[u]

    # Reduce phase
    base = (beta * dangling_sum + (1-beta)) / n
    new_rank = {}
    all_nodes = set(u for u,v in edges) | set(v for u,v in edges)
    for node in all_nodes:
        new_rank[node] = intermediate.get(node,0.0) + base
    return new_rank

edges = [(0,1),(0,2),(1,3),(2,1),(2,3),(3,0)]
rank = {i:0.25 for i in range(4)}
for it in range(30):
    new_rank = mapreduce_pagerank_step(edges, rank, 4)
    diff = sum(abs(new_rank.get(i,0)-rank.get(i,0)) for i in range(4))
    rank = new_rank
    if diff < 1e-8: print(f"MR PageRank converged at iteration {it+1}"); break
print({i:f'{v:.4f}' for i,v in rank.items()})

{0: '0.2972', 1: '0.2334', 2: '0.1638', 3: '0.3056'}


## I/O Cost Analysis

| Quantity | Value | Notes |
|----------|-------|-------|
| Web pages $n$ | $5\times10^{10}$ | Estimated crawlable web |
| Web links $m$ | $10^{12}$ | ~20 links/page avg |
| Rank vector size | $5\times10^{10} \times 8$ B = 400 GB | Must fit distributed RAM |
| Edge list size | $10^{12} \times 16$ B = 16 TB | Read once per iteration |
| Iterations to converge | ~50-100 | Depends on $\beta$ |
| Total I/O | ~1-2 PB | With block-stripe, single pass per iter |

**Google's approach (circa 2004):** 10,000 machines × 10 GB RAM = 100 TB aggregate RAM.
PageRank was recomputed monthly; later daily; now continuously (personalized).